In [2]:
# ================================================================
# DATA PIPELINE — FINAL
# ================================================================

# ================================
# 🔹 1. Mount Google Drive
# ================================
from google.colab import drive
drive.mount('/content/drive')

# ================================
# 🔹 2. Paths
# ================================
import os

DATA_PATH        = "/content/drive/MyDrive/SER_Project/"
RAVDESS_FOLDERS  = [DATA_PATH + "Ravdess_Song/", DATA_PATH + "Ravdess_Speech/"]
TESS_FOLDER      = DATA_PATH + "Tess/"

SAVE_X_PATH      = DATA_PATH + "X_fixed_final.npy"
SAVE_Y_PATH      = DATA_PATH + "y_fixed_final.npy"

# ================================
# 🔹 3. Imports
# ================================
import librosa
import numpy as np
from collections import Counter

# ================================
# 🔹 4. Emotion Labels (MASTER)
# ================================
EMOTION_LABELS = [
    "neutral",   # 0
    "calm",      # 1
    "happy",     # 2
    "sad",       # 3
    "angry",     # 4
    "fearful",   # 5
    "surprise",  # 6
    "disgust",   # 7
]

LABEL2IDX = {label: idx for idx, label in enumerate(EMOTION_LABELS)}

# ================================
# 🔹 5. Dataset Label Maps
# ================================
emotion_map_ravdess = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprise",
}

emotion_map_tess = {
    "neutral": "neutral",
    "happy": "happy",
    "sad": "sad",
    "angry": "angry",
    "fear": "fearful",
    "disgust": "disgust",
    "pleasant_surprise": "surprise",
}

# ================================
# 🔹 6. Feature Extraction
# ================================
def extract_features(file_path, sr=16000, n_mels=128):
    audio, _ = librosa.load(file_path, sr=sr)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=1024,
        hop_length=512,
        n_mels=n_mels
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    # FIX: (mel, time) → (time, mel)
    return mel_db.T

# ================================
# 🔹 7. Padding
# ================================
def pad_features(features, max_len=128):
    T = features.shape[0]

    if T < max_len:
        features = np.pad(features, ((0, max_len - T), (0, 0)))
    else:
        features = features[:max_len, :]

    return features

# ================================
# 🔹 8. Load RAVDESS
# ================================
def load_ravdess(folder_list):
    X, y = [], []
    skipped = 0

    for base_folder in folder_list:
        for actor in os.listdir(base_folder):
            actor_path = os.path.join(base_folder, actor)

            if os.path.isdir(actor_path):
                for file in os.listdir(actor_path):
                    if file.endswith(".wav"):
                        parts = file.split("-")
                        if len(parts) < 3:
                            skipped += 1
                            continue

                        emotion_code = parts[2]
                        label = emotion_map_ravdess.get(emotion_code)

                        if label in LABEL2IDX:
                            path = os.path.join(actor_path, file)
                            feat = pad_features(extract_features(path))
                            X.append(feat)
                            y.append(LABEL2IDX[label])
                        else:
                            skipped += 1

    print(f"RAVDESS: {len(X)} loaded, {skipped} skipped")
    return X, y

# ================================
# 🔹 9. Load TESS (FIXED)
# ================================
def load_tess(folder):
    X, y = [], []
    skipped = 0

    for root, _, files in os.walk(folder):
        for file in files:
            if file.endswith(".wav"):
                name = os.path.splitext(file)[0]

                # ✅ FIX: take LAST token only
                label_raw = name.split("_")[-1].lower()

                label = emotion_map_tess.get(label_raw)

                if label in LABEL2IDX:
                    path = os.path.join(root, file)
                    feat = pad_features(extract_features(path))
                    X.append(feat)
                    y.append(LABEL2IDX[label])
                else:
                    skipped += 1
                    print("Skipped:", label_raw)  # debug

    print(f"TESS   : {len(X)} loaded, {skipped} skipped")
    return X, y

# ================================
# 🔹 10. MAIN PIPELINE
# ================================
if os.path.exists(SAVE_X_PATH):
    print("✅ Loading preprocessed data...")
    X = np.load(SAVE_X_PATH)
    y = np.load(SAVE_Y_PATH)

else:
    print("⚙️ Processing data...")

    X_r, y_r = load_ravdess(RAVDESS_FOLDERS)
    X_t, y_t = load_tess(TESS_FOLDER)

    X = np.array(X_r + X_t, dtype=np.float32)
    y = np.array(y_r + y_t, dtype=np.int64)

    print(f"\nTotal samples: {len(X)}")

    np.save(SAVE_X_PATH, X)
    np.save(SAVE_Y_PATH, y)

# ================================
# 🔹 11. VERIFICATION
# ================================
print("\n===== VERIFICATION =====")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Range:", X.min(), X.max())

print("\nClass Distribution:")
counts = Counter(y.tolist())
for i, label in enumerate(EMOTION_LABELS):
    print(f"{label:<10s}: {counts.get(i,0)}")

print("\n✅ Ready for training!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚙️ Processing data...
RAVDESS: 2452 loaded, 0 skipped
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: disgust (1)
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skipped: ps
Skip